In [5]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class KvasirDataset(Dataset):
    def __init__(self, data_dir, split_file, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        
        # Read the list of image IDs from train.txt or val.txt
        split_path = os.path.join(data_dir, split_file)
        with open(split_path, "r") as f:
            self.image_ids = [line.strip() for line in f if line.strip()]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        
        img_path = os.path.join(self.data_dir, "images", f"{img_id}.jpg")
        mask_path = os.path.join(self.data_dir, "masks", f"{img_id}.jpg")

        # Load image (RGB) and mask (Grayscale)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Make the mask binary (0 or 1)
        mask = (mask > 128).astype(np.float32)

        # Apply resizing and normalization
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]
        else:
            image = ToTensorV2()(image=image)["image"].float() / 255.0
            mask = torch.tensor(mask, dtype=torch.float32)

        # PyTorch expects masks to have a channel dimension: [1, H, W]
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
            
        return image, mask

def get_dataloaders(data_dir="data", batch_size=4, img_size=256):
    # Standard medical image augmentations
    train_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    val_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    train_dataset = KvasirDataset(data_dir, "train.txt", transform=train_transform)
    val_dataset = KvasirDataset(data_dir, "val.txt", transform=val_transform)
    
    # num_workers=0 prevents freezing issues in Windows Jupyter notebooks
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    return train_loader, val_loader

# --- Quick Test ---
print("Loading data...")
train_loader, val_loader = get_dataloaders(data_dir="data", batch_size=4)
images, masks = next(iter(train_loader))

print("Data loaded successfully!")
print(f"Images shape: {images.shape}")
print(f"Masks shape: {masks.shape}")

Loading data...
Data loaded successfully!
Images shape: torch.Size([4, 3, 256, 256])
Masks shape: torch.Size([4, 1, 256, 256])


In [6]:
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1):
        super().__init__()
        
        # Encoder (Downsampling)
        self.enc1 = ConvBlock(in_channels, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Bottleneck
        self.bottleneck = ConvBlock(512, 1024)
        
        # Decoder (Upsampling)
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(1024, 512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(256, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(128, 64)
        
        # Final 1x1 convolution to map to a binary mask
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # Down
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e4))
        
        # Up (concatenating skip connections from the encoder)
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        return self.final_conv(d1)

# --- Quick Test ---
print("Initializing Standard U-Net...")
unet_model = UNet(in_channels=3, num_classes=1)

# Pass the 'images' variable we created in Cell 1 through the model
unet_preds = unet_model(images) 

print("U-Net test successful!")
print(f"Predictions shape: {unet_preds.shape}")

Initializing Standard U-Net...
U-Net test successful!
Predictions shape: torch.Size([4, 1, 256, 256])


In [4]:
import torch
import torch.nn as nn
import numpy as np

# ==========================================
# 1. EXACT MATCHING ARCHITECTURE DEFINITION
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class AttentionGate(nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.w_g = nn.Sequential(
            nn.Conv2d(f_g, f_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(f_int)
        )
        self.w_x = nn.Sequential(
            nn.Conv2d(f_l, f_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(f_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(f_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.w_g(g)
        x1 = self.w_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1):
        super().__init__()
        
        self.enc1 = ConvBlock(in_channels, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = ConvBlock(512, 1024)
        
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.att4 = AttentionGate(f_g=512, f_l=512, f_int=256)
        self.dec4 = ConvBlock(1024, 512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.att3 = AttentionGate(f_g=256, f_l=256, f_int=128)
        self.dec3 = ConvBlock(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.att2 = AttentionGate(f_g=128, f_l=128, f_int=64)
        self.dec2 = ConvBlock(256, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.att1 = AttentionGate(f_g=64, f_l=64, f_int=32)
        self.dec1 = ConvBlock(128, 64)
        
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        b = self.bottleneck(self.pool(e4))
        
        d4 = self.up4(b)
        x4 = self.att4(g=d4, x=e4)
        d4 = self.dec4(torch.cat([d4, x4], dim=1))
        
        d3 = self.up3(d4)
        x3 = self.att3(g=d3, x=e3)
        d3 = self.dec3(torch.cat([d3, x3], dim=1))
        
        d2 = self.up2(d3)
        x2 = self.att2(g=d2, x=e2)
        d2 = self.dec2(torch.cat([d2, x2], dim=1))
        
        d1 = self.up1(d2)
        x1 = self.att1(g=d1, x=e1)
        d1 = self.dec1(torch.cat([d1, x1], dim=1))
        
        return self.final_conv(d1)

# ==========================================
# 2. EVALUATION PIPELINE
# ==========================================
def evaluate_full_metrics(model, loader, device):
    model.eval()
    total_dice, total_iou, total_prec, total_rec = 0.0, 0.0, 0.0, 0.0
    hd95_list = []
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            
            intersection = (preds * masks).sum()
            total_dice += (2.0 * intersection / (preds.sum() + masks.sum() + 1e-7)).item()
            total_iou += (intersection / (preds.sum() + masks.sum() - intersection + 1e-7)).item()
            
            prec, rec = calculate_precision_recall(preds, masks)
            total_prec += prec
            total_rec += rec
            
            for p, m in zip(preds, masks):
                val_hd95 = calculate_hd95(p, m)
                if not np.isnan(val_hd95):
                    hd95_list.append(val_hd95)
                    
    num_batches = len(loader)
    return {
        "Dice": total_dice / num_batches,
        "IoU": total_iou / num_batches,
        "Precision": total_prec / num_batches,
        "Recall": total_rec / num_batches,
        "HD95": np.mean(hd95_list) if len(hd95_list) > 0 else np.nan
    }

# ==========================================
# 3. LOAD WEIGHTS & EVALUATE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
att_model = AttentionUNet(in_channels=3, num_classes=1).to(device)
att_model.load_state_dict(torch.load("best_att_unet.pth", map_location=device, weights_only=True))

print("Running full evaluation on best Attention U-Net...")
results = evaluate_full_metrics(att_model, val_loader, device)

print("\n" + "="*38)
print(" TASK 3: ATTENTION U-NET METRICS")
print("="*38)
for metric, score in results.items():
    print(f"{metric:<12}: {score:.4f}")
print("="*38)

Running full evaluation on best Attention U-Net...


NameError: name 'val_loader' is not defined

In [11]:
print("running..")
from tqdm import tqdm # Jupyter-friendly progress bars

class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, preds, targets):
        # 1. Binary Cross Entropy Loss
        bce_loss = self.bce(preds, targets)
        
        # 2. Dice Loss
        probs = torch.sigmoid(preds).view(-1)
        targets_flat = targets.view(-1)
        
        intersection = (probs * targets_flat).sum()
        dice_loss = 1 - (2.0 * intersection + self.smooth) / (probs.sum() + targets_flat.sum() + self.smooth)
        
        return bce_loss + dice_loss

def calculate_metrics(preds, targets, smooth=1e-6):
    probs = torch.sigmoid(preds)
    preds_bin = (probs > 0.5).float().view(-1)
    targets_flat = targets.view(-1)

    intersection = (preds_bin * targets_flat).sum().item()
    union = preds_bin.sum().item() + targets_flat.sum().item()

    dice = (2.0 * intersection + smooth) / (union + smooth)
    iou = (intersection + smooth) / (union - intersection + smooth)
    
    return dice, iou

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    # tqdm gives us a nice progress bar in Jupyter
    for images, masks in tqdm(loader, desc="Training", leave=False):
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total_dice, total_iou = 0.0, 0.0

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Validation", leave=False):
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_loss += loss.item() * images.size(0)
            d, i = calculate_metrics(outputs, masks)
            total_dice += d * images.size(0)
            total_iou += i * images.size(0)

    return (
        running_loss / len(loader.dataset),
        total_dice / len(loader.dataset),
        total_iou / len(loader.dataset)
    )
print("finished")

running..
finished


In [12]:
print("running..")
from tqdm.notebook import tqdm # Jupyter-friendly progress bars

class DiceBCELoss(nn.Module):
    """
    Standard loss for medical image segmentation.
    BCE handles pixel-by-pixel accuracy, while Dice handles overall shape overlap.
    """
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, preds, targets):
        # 1. Binary Cross Entropy Loss
        bce_loss = self.bce(preds, targets)
        
        # 2. Dice Loss
        probs = torch.sigmoid(preds).view(-1)
        targets_flat = targets.view(-1)
        
        intersection = (probs * targets_flat).sum()
        dice_loss = 1 - (2.0 * intersection + self.smooth) / (probs.sum() + targets_flat.sum() + self.smooth)
        
        return bce_loss + dice_loss

def calculate_metrics(preds, targets, smooth=1e-6):
    """Calculates Dice Score and IoU for validation tracking."""
    probs = torch.sigmoid(preds)
    preds_bin = (probs > 0.5).float().view(-1)
    targets_flat = targets.view(-1)

    intersection = (preds_bin * targets_flat).sum().item()
    union = preds_bin.sum().item() + targets_flat.sum().item()

    dice = (2.0 * intersection + smooth) / (union + smooth)
    iou = (intersection + smooth) / (union - intersection + smooth)
    
    return dice, iou

def train(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    # tqdm gives us a nice progress bar in Jupyter
    for images, masks in tqdm(loader, desc="Training", leave=False):
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total_dice, total_iou = 0.0, 0.0

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Validation", leave=False):
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_loss += loss.item() * images.size(0)
            d, i = calculate_metrics(outputs, masks)
            total_dice += d * images.size(0)
            total_iou += i * images.size(0)

    return (
        running_loss / len(loader.dataset),
        total_dice / len(loader.dataset),
        total_iou / len(loader.dataset)
    )
print("finished")

running..
finished


In [ ]:
import torch

print("running..")

# 0. Define the device and loss function just in case Jupyter forgot them
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = DiceBCELoss()

# 1. Initialize Attention U-Net
att_model = AttentionUNet(in_channels=3, num_classes=1).to(device)
att_save_path = "best_att_unet.pth"

# 2. We need a NEW optimizer specifically for this model
att_optimizer = torch.optim.Adam(att_model.parameters(), lr=1e-4)

# 3. Run 1 Test Epoch
epochs = 60
best_att_dice = 0.0

print("Starting training for Attention U-Net...")
for epoch in range(1, epochs + 1):
    train_loss = train_one_epoch(att_model, train_loader, criterion, att_optimizer, device)
    val_loss, val_dice, val_iou = validate(att_model, val_loader, criterion, device)

    print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}")

    if val_dice > best_att_dice:
        best_att_dice = val_dice
        torch.save(att_model.state_dict(), att_save_path)
        print(f" -> Saved new best Attention model with Dice: {best_att_dice:.4f}")
        
print("finished")

running..
Starting training for Attention U-Net...


Training:   0%|          | 0/220 [00:00<?, ?it/s]

Validation:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01/60 | Train Loss: 1.0573 | Val Loss: 1.0120 | Val Dice: 0.4975 | Val IoU: 0.3407
 -> Saved new best Attention model with Dice: 0.4975


Training:   0%|          | 0/220 [00:00<?, ?it/s]

...

Epoch 58/60 | Train Loss: 0.2394 | Val Loss: 0.3667 | Val Dice: 0.8281 | Val IoU: 0.7179
 -> Saved new best Attention model with Dice: 0.8281


Training:   0%|          | 0/220 [00:00<?, ?it/s]

Validation:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 59/60 | Train Loss: 0.2283 | Val Loss: 0.4129 | Val Dice: 0.8047 | Val IoU: 0.6893


Training:   0%|          | 0/220 [00:00<?, ?it/s]

In [1]:
print("finished")
import torch
print("GPU Available:", torch.cuda.is_available())

import torch
print("GPU Available:", torch.cuda.is_available())

finished
GPU Available: True
GPU Available: True


In [1]:
import torch
import torch.nn as nn
import numpy as np

# ==========================================
# 1. ATTENTION U-NET ARCHITECTURE DEFINITION
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1):
        super().__init__()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv1 = ConvBlock(in_channels, 64)
        self.conv2 = ConvBlock(64, 128)
        self.conv3 = ConvBlock(128, 256)
        self.conv4 = ConvBlock(256, 512)
        self.conv5 = ConvBlock(512, 1024)

        self.up5 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.att5 = AttentionGate(F_g=512, F_l=512, F_int=256)
        self.up_conv5 = ConvBlock(1024, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.att4 = AttentionGate(F_g=256, F_l=256, F_int=128)
        self.up_conv4 = ConvBlock(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.att3 = AttentionGate(F_g=128, F_l=128, F_int=64)
        self.up_conv3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.att2 = AttentionGate(F_g=64, F_l=64, F_int=32)
        self.up_conv2 = ConvBlock(128, 64)

        self.out_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.conv1(x)
        e2 = self.conv2(self.maxpool(e1))
        e3 = self.conv3(self.maxpool(e2))
        e4 = self.conv4(self.maxpool(e3))
        e5 = self.conv5(self.maxpool(e4))

        d5 = self.up5(e5)
        x4 = self.att5(g=d5, x=e4)
        d5 = torch.cat((x4, d5), dim=1)
        d5 = self.up_conv5(d5)

        d4 = self.up4(d5)
        x3 = self.att4(g=d4, x=e3)
        d4 = torch.cat((x3, d4), dim=1)
        d4 = self.up_conv4(d4)

        d3 = self.up3(d4)
        x2 = self.att3(g=d3, x=e2)
        d3 = torch.cat((x2, d3), dim=1)
        d3 = self.up_conv3(d3)

        d2 = self.up2(d3)
        x1 = self.att2(g=d2, x=e1)
        d2 = torch.cat((x1, d2), dim=1)
        d2 = self.up_conv2(d2)

        return self.out_conv(d2)

# ==========================================
# 2. EVALUATION PIPELINE
# ==========================================
def evaluate_full_metrics(model, loader, device):
    model.eval()
    total_dice, total_iou, total_prec, total_rec = 0.0, 0.0, 0.0, 0.0
    hd95_list = []
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            
            intersection = (preds * masks).sum()
            total_dice += (2.0 * intersection / (preds.sum() + masks.sum() + 1e-7)).item()
            total_iou += (intersection / (preds.sum() + masks.sum() - intersection + 1e-7)).item()
            
            prec, rec = calculate_precision_recall(preds, masks)
            total_prec += prec
            total_rec += rec
            
            for p, m in zip(preds, masks):
                val_hd95 = calculate_hd95(p, m)
                if not np.isnan(val_hd95):
                    hd95_list.append(val_hd95)
                    
    num_batches = len(loader)
    return {
        "Dice": total_dice / num_batches,
        "IoU": total_iou / num_batches,
        "Precision": total_prec / num_batches,
        "Recall": total_rec / num_batches,
        "HD95": np.mean(hd95_list) if len(hd95_list) > 0 else np.nan
    }

# ==========================================
# 3. LOAD WEIGHTS & EVALUATE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
att_model = AttentionUNet(in_channels=3, num_classes=1).to(device)
att_model.load_state_dict(torch.load("best_att_unet.pth", map_location=device))

results = evaluate_full_metrics(att_model, val_loader, device)

print("="*35)
print(" TASK 3: OFFICIAL EVALUATION METRICS")
print("="*35)
for metric, score in results.items():
    print(f"{metric:<12}: {score:.4f}")
print("="*35)

NameError: name 'AttentionUNet' is not defined

In [2]:
print("working...")
import numpy as np
from scipy.spatial import cKDTree

def calculate_precision_recall(preds, masks):
    # Flatten the tensors
    preds_flat = preds.view(-1)
    masks_flat = masks.view(-1)
    
    # Calculate True Positives, False Positives, and False Negatives
    tp = (preds_flat * masks_flat).sum().item()
    fp = (preds_flat * (1 - masks_flat)).sum().item()
    fn = ((1 - preds_flat) * masks_flat).sum().item()
    
    # Add a tiny epsilon to prevent dividing by zero
    epsilon = 1e-7
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    
    return precision, recall

def calculate_hd95(preds, masks):
    # Convert PyTorch tensors to Numpy arrays
    preds_np = preds.cpu().numpy().squeeze()
    masks_np = masks.cpu().numpy().squeeze()
    
    # If either mask is completely empty, return a default bad value (e.g., NaN)
    if np.sum(preds_np) == 0 or np.sum(masks_np) == 0:
        return np.nan
        
    # Get the coordinates of the boundaries (pixels > 0)
    pred_coords = np.array(np.nonzero(preds_np)).T
    mask_coords = np.array(np.nonzero(masks_np)).T
    
    # Build KD-trees for fast distance calculation
    pred_tree = cKDTree(pred_coords)
    mask_tree = cKDTree(mask_coords)
    
    # Compute the shortest distance from every point in Pred to the nearest point in Mask, and vice versa
    dist_pred_to_mask, _ = mask_tree.query(pred_coords)
    dist_mask_to_pred, _ = pred_tree.query(mask_coords)
    
    # Combine the distances and find the 95th percentile
    all_distances = np.concatenate([dist_pred_to_mask, dist_mask_to_pred])
    hd95 = np.percentile(all_distances, 95)
    
    return hd95
print("finished")

working...
finished


In [2]:

import matplotlib.pyplot as plt
import numpy as np

def visualize_predictions(model, loader, device, num_images=3):
    model.eval() # Set model to evaluation mode
    
    # Grab just one batch of data from the validation loader
    images, masks = next(iter(loader))
    images = images.to(device)
    
    # Run the images through the model
    with torch.no_grad():
        preds = model(images)
        preds = torch.sigmoid(preds)  # Convert outputs to probabilities (0 to 1)
        preds = (preds > 0.5).float() # Convert to strict black and white (binary mask)
        
    # Bring the data back to the CPU so matplotlib can draw it
    images = images.cpu().numpy()
    masks = masks.cpu().numpy()
    preds = preds.cpu().numpy()
    
    fig, axes = plt.subplots(num_images, 3, figsize=(12, 4 * num_images))
    
    for i in range(num_images):
        # 1. Original Image: convert from (Channels, Height, Width) to (Height, Width, Channels)
        img = np.transpose(images[i], (1, 2, 0))
        # Normalize display just in case pixels are outside standard ranges
        img = (img - img.min()) / (img.max() - img.min())
        
        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Original Image")
        axes[i, 0].axis("off")
        
        # 2. Ground Truth Mask
        axes[i, 1].imshow(masks[i].squeeze(), cmap='gray')
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis("off")
        
        # 3. Model Prediction
        axes[i, 2].imshow(preds[i].squeeze(), cmap='gray')
        axes[i, 2].set_title("Attention U-Net Prediction")
        axes[i, 2].axis("off")
        
    plt.tight_layout()
    plt.show()

# Run it!
# 1. Load the absolute best weights saved during training
att_model.load_state_dict(torch.load(att_save_path))

# 2. Run the visualization with an updated label
print("Visualizing results for the Best Attention U-Net ...")
visualize_predictions(att_model, val_loader, device)

NameError: name 'att_model' is not defined

In [3]:
import torch
import torch.nn as nn
import numpy as np

# 1. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Instantiate using your exact AttentionUNet definition
att_model = AttentionUNet(in_channels=3, num_classes=1).to(device)

# 3. Load the saved weights
att_model.load_state_dict(torch.load("best_att_unet.pth", map_location=device, weights_only=True))

# 4. Run Task 3 Evaluation across all five metrics
print("Running full evaluation on best Attention U-Net...")
results = evaluate_full_metrics(att_model, val_loader, device)

# 5. Display the comparison metrics
print("\n" + "="*38)
print(" TASK 3: ATTENTION U-NET METRICS")
print("="*38)
for metric, score in results.items():
    print(f"{metric:<12}: {score:.4f}")
print("="*38)

RuntimeError: Error(s) in loading state_dict for AttentionUNet:
	Missing key(s) in state_dict: "conv1.conv.0.weight", "conv1.conv.1.weight", "conv1.conv.1.bias", "conv1.conv.1.running_mean", "conv1.conv.1.running_var", "conv1.conv.3.weight", "conv1.conv.4.weight", "conv1.conv.4.bias", "conv1.conv.4.running_mean", "conv1.conv.4.running_var", "conv2.conv.0.weight", "conv2.conv.1.weight", "conv2.conv.1.bias", "conv2.conv.1.running_mean", "conv2.conv.1.running_var", "conv2.conv.3.weight", "conv2.conv.4.weight", "conv2.conv.4.bias", "conv2.conv.4.running_mean", "conv2.conv.4.running_var", "conv3.conv.0.weight", "conv3.conv.1.weight", "conv3.conv.1.bias", "conv3.conv.1.running_mean", "conv3.conv.1.running_var", "conv3.conv.3.weight", "conv3.conv.4.weight", "conv3.conv.4.bias", "conv3.conv.4.running_mean", "conv3.conv.4.running_var", "conv4.conv.0.weight", "conv4.conv.1.weight", "conv4.conv.1.bias", "conv4.conv.1.running_mean", "conv4.conv.1.running_var", "conv4.conv.3.weight", "conv4.conv.4.